<a href="https://colab.research.google.com/github/brownt47/NBA_DraftKings/blob/main/LSTM_Model_NBA_Only_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Loading and Preperation

##Load all csv files needed
####player_data_cleaned (Season player data)
####DKSalaries-[contest] (Draft Kings player data for contest)
####injury_report (Players who are injury out)

In [ ]:
import pandas as pd

# Load the data
data = pd.read_csv('/content/player_data_cleaned_02-11-2025.csv')

data = data.sort_values(by=['PLAYER', 'GAME DATE'])

##Load and clean Season Data for each player

In [ ]:
data.columns

Index(['PLAYER', 'TEAM', 'MATCH UP', 'GAME DATE', 'W/L', 'MIN', 'PTS', 'FGM',
       'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', '+/-', 'First', 'Last',
       'Suffix', 'KEY', 'PER', 'DoubleDouble', 'TripleDouble', 'Home_Away',
       'Opponent', 'Opponent_ATL', 'Opponent_BKN', 'Opponent_BOS',
       'Opponent_CHA', 'Opponent_CHI', 'Opponent_CLE', 'Opponent_DAL',
       'Opponent_DEN', 'Opponent_DET', 'Opponent_GSW', 'Opponent_HOU',
       'Opponent_IND', 'Opponent_LAC', 'Opponent_LAL', 'Opponent_MEM',
       'Opponent_MIA', 'Opponent_MIL', 'Opponent_MIN', 'Opponent_NOP',
       'Opponent_NYK', 'Opponent_OKC', 'Opponent_ORL', 'Opponent_PHI',
       'Opponent_PHX', 'Opponent_POR', 'Opponent_SAC', 'Opponent_SAS',
       'Opponent_TOR', 'Opponent_UTA', 'Opponent_WAS', 'FP'],
      dtype='object')

In [ ]:
features = ['MIN', 'PTS', 'FGM',
       'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', '+/-', 'PER', 'DoubleDouble',
       'DoubleDouble', 'TripleDouble',
       'Home_Away', 'Opponent_ATL', 'Opponent_BKN', 'Opponent_BOS',
       'Opponent_CHA', 'Opponent_CHI', 'Opponent_CLE', 'Opponent_DAL',
       'Opponent_DEN', 'Opponent_DET', 'Opponent_GSW', 'Opponent_HOU',
       'Opponent_IND', 'Opponent_LAC', 'Opponent_LAL', 'Opponent_MEM',
       'Opponent_MIA', 'Opponent_MIL', 'Opponent_MIN', 'Opponent_NOP',
       'Opponent_NYK', 'Opponent_OKC', 'Opponent_ORL', 'Opponent_PHI',
       'Opponent_PHX', 'Opponent_POR', 'Opponent_SAC', 'Opponent_SAS',
       'Opponent_TOR', 'Opponent_UTA', 'Opponent_WAS']

target = 'FP'  # Fantasy Points

#Scale data to enter into LSTM Model

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Separate scalers for features and target
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Scale features and target separately
data[features] = feature_scaler.fit_transform(data[features])
data[[target]] = target_scaler.fit_transform(data[[target]])

#Build and Train Model

In [ ]:
# Function to create sequences for LSTM
def create_sequences(df, feature_cols, target_col, sequence_length):
    sequences = []
    targets = []
    players = df['KEY'].unique()

    for player in players:
        player_data = df[df['KEY'] == player][feature_cols + [target_col]].values
        if len(player_data) > sequence_length:
            for i in range(len(player_data) - sequence_length):
                sequences.append(player_data[i:i+sequence_length, :-1])  # Input features
                targets.append(player_data[i+sequence_length, -1])       # Target value

    return np.array(sequences), np.array(targets)

# Prepare Data (Make sure your 'data', 'features', and 'target' variables are defined)
def prepare_data(sequence_length):
    X, y = create_sequences(data, features, target, sequence_length)
    X = X.reshape((X.shape[0], X.shape[1], len(features)))
    return X, y

# Best Hyperparameters from Tuning
best_params = {
    'sequence_length': 7,
    'num_lstm_layers': 1,
    'lstm_units_1': 512,
    'dropout_lstm_1': 0.2,
    'dense_units': 32,
    'dense_activation': 'tanh',
    'learning_rate': 0.0001
}

# Prepare data with the best sequence length
X, y = prepare_data(best_params['sequence_length'])

# TimeSeriesSplit Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Save the best model based on validation loss
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

# Build the model with the best hyperparameters
model = Sequential()
model.add(LSTM(units=best_params['lstm_units_1'], activation='tanh', input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(best_params['dropout_lstm_1']))
model.add(Dense(best_params['dense_units'], activation=best_params['dense_activation']))
model.add(Dense(1))  # Predicting Fantasy Points

# Compile the model
model.compile(optimizer=Adam(learning_rate=best_params['learning_rate']), loss='mse')

# Train the model with both EarlyStopping and ModelCheckpoint
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, model_checkpoint]
)

# Load the best model saved during training
best_model = load_model('best_model.keras')

# Predict and evaluate using the best model
predictions = best_model.predict(X_test)
predictions_rescaled = target_scaler.inverse_transform(predictions)

# Inverse transform the actual target values for comparison
y_test_rescaled = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluate the model
mse = mean_squared_error(y_test_rescaled, predictions_rescaled)
mae = mean_absolute_error(y_test_rescaled, predictions_rescaled)
r2 = r2_score(y_test_rescaled, predictions_rescaled)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"R-squared: {r2}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 41s 111ms/step - loss: 0.0128 - val_loss: 0.0105
Epoch 2/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 33s 88ms/step - loss: 0.0108 - val_loss: 0.0099
Epoch 3/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 84ms/step - loss: 0.0103 - val_loss: 0.0102
Epoch 4/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 81ms/step - loss: 0.0103 - val_loss: 0.0099
Epoch 5/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - loss: 0.0098 - val_loss: 0.0099
Epoch 6/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 79ms/step - loss: 0.0103 - val_loss: 0.0099
Epoch 7/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 41s 80ms/step - loss: 0.0099 - val_loss: 0.0099
Epoch 8/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 41s 81ms/step - loss: 0.0097 - val_loss: 0.0098
Epoch 9/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 42s 84ms/step - loss: 0.0098 - val_loss: 0.0103
Epoch 10/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 80ms/step - loss: 0.0100 - val_loss: 0.0103
Epoch 11/50
351/351 ━━━━━━━━━━━━━━━━━━━━ 44s 89ms/step - loss: 0.0094 - val_loss: 0.0098
Epoch 12/50
351/351 ━━━━━━━━━

In [ ]:
predictions

array([[0.24987641],
       [0.26131436],
       [0.25559354],
       ...,
       [0.4515108 ],
       [0.45566824],
       [0.4729542 ]], dtype=float32)

In [ ]:
nextgame= zeros_array = np.zeros(len(features))
nextgame

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0.])

In [ ]:
sequence_length = 7

In [ ]:
# Get the list of unique players
players = data['KEY'].unique()

# Create a dictionary to hold predictions
next_game_predictions = {}

# Loop through each player to prepare data and make predictions
for player in players:
    player_data = data[data['KEY'] == player].sort_values(by='GAME DATE')

    # Ensure there are enough past games to create a sequence
    if len(player_data) >= sequence_length:
        # Get the last N games for the player
        recent_games = player_data[features].values[-sequence_length:]

        # Reshape for LSTM input: (1 sample, sequence_length, number of features)
        input_sequence = recent_games.reshape((1, sequence_length, len(features)))

        # Predict the next game's fantasy points (SCALED)
        prediction_scaled = model.predict(input_sequence)

        # Inverse transform to get actual fantasy points (OLD METHOD causing issues)
        prediction = target_scaler.inverse_transform(
            np.hstack((np.zeros((1, len(features))), prediction_scaled))
        )[:, -1][0]

        # Store the prediction in the dictionary
        next_game_predictions[player] = prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━

In [ ]:
# Convert predictions to a DataFrame for better display
predictions_df = pd.DataFrame(list(next_game_predictions.items()), columns=['KEY', 'PredictedFantasyPoints'])

# Display the rescaled predictions
#predictions_df

In [ ]:
predictions_df[predictions_df['KEY']=='nikjokic']

,KEY,PredictedFantasyPoints
357,nikjokic,59.196133


In [ ]:
from datetime import datetime
import pytz

# Get current UTC time
utc_dt = datetime.utcnow().replace(tzinfo=pytz.utc)

# Convert UTC to Eastern Time
eastern = pytz.timezone('US/Eastern')
eastern_dt = utc_dt.astimezone(eastern)

# Display conversions
print(f"UTC Time: {utc_dt}")
print(f"Converted to Eastern Time: {eastern_dt}")

# Use Eastern Time for the filename
timestamp = eastern_dt.strftime('%m_%d_%Y')
output_filename = f'DK_LSTM_Predictions_{timestamp}.csv'

# Export to CSV with Eastern Time in the filename
predictions_df.to_csv(output_filename, index=False)

# Confirmation message
print(f"Exported to {output_filename} successfully!")


UTC Time: 2025-02-11 17:31:16.870253+00:00
Converted to Eastern Time: 2025-02-11 12:31:16.870253-05:00
Exported to DK_LSTM_Predictions_02_11_2025.csv successfully!
